# Inference Script

This notebook will run inference of the airgun detector in the data from 2025 with the selected airgun model, after running the the evaluate ensemble scripts the conclusion is that models are indistinguishable from each other from step 3 onwards so we will use the step 7 model. Additionally the best enesemble was always the Resnet+Perch+Effnet. We will use that model. 

### Location of the data

The data for 2025 is located in 
- I:\Johannes_Helena\HABITATWal\2025\STHH2
- I:\Johannes_Helena\HABITATWal\2025\STHH3
- I:\Johannes_Helena\HABITATWal\2025\STHH4
- I:\Johannes_Helena\HABITATWal\2025\STHH5
- I:\Schalldaten_HABITATWal\2025\STHH1\AMAR


In [6]:
# Other utilities and packages
from opensoundscape.ml.cnn import load_model
from opensoundscape import Audio
import bioacoustics_model_zoo as bmz
import joblib
import opensoundscape
import torch
from pathlib import Path
import numpy as np
import pandas as pd
from glob import glob
import subprocess
#set up plotting
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize']=[15,5] #for large visuals
%config InlineBackend.figure_format = 'retina'

# Define the root folder with the current files to analyze
root = r"/mnt/i/Schalldaten_HABITATWal/2025/STHH1/AMAR/batch4"

resnet= load_model("annotation_step7/resnet/best.model")
effnet = load_model("annotation_step7/effnet/best.model")
perch = bmz.Perch2ONNX(headless=True)
shallow = joblib.load("annotation_step7/shallow_perch/shallow_classifier.joblib")





/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/preprocess/overlay.py:126: UserWarning: Overlay class's .overlay_df will be None after loading from dict and `.criterion_fn` will be always_true(). Reset these attributes and set .bypass to False to use Overlay after loading with from_dict().
  warnings.warn(
/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/preprocess/overlay.py:126: UserWarning: Overlay class's .overlay_df will be None after loading from dict and `.criterion_fn` will be always_true(). Reset these attributes and set .bypass to False to use Overlay after loading with from_dict().
  warnings.warn(


In [7]:
all_files = sorted(glob(str(Path(root) / "*.wav"), recursive=True))
len(all_files)
# # select only the files that are not already processed
# #read the two chunks that are processed
# chunk1 = pd.read_csv("/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/detections_airgun_2025/chunk_0000.csv")
# chunk2 = pd.read_csv("/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/detections_airgun_2025/chunk_0001.csv")
# chunk3 = pd.read_csv("/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/detections_airgun_2025/chunk_0002.csv")
# chunk4 = pd.read_csv("/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/detections_airgun_2025/chunk_0003.csv")
# processed = set(pd.concat([chunk1, chunk2, chunk3, chunk4])["file"])
# all_files = [f for f in all_files if f not in processed]
# len(all_files)

2000

In [ ]:
from pathlib import Path
from opensoundscape.utils import make_clip_df
out_dir = Path("/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/detections_airgun_2025/batch4"); out_dir.mkdir(exist_ok=True)
CHUNK = 100                                         # files per chunk

for c in range(0, len(all_files), CHUNK):
    files = all_files[c:c + CHUNK]
    out = out_dir / f"chunk_{Path(files[0]).stem}.csv"     # named after the first file, stable across reruns
    if out.exists():
        continue                                   # already done
    clips = make_clip_df(files, clip_duration=5)
    if "start_time" in clips.columns:
        clips = clips.set_index(["start_time", "end_time"], append=True)

    s = pd.DataFrame(index=clips.index)
    s["resnet"] = resnet.predict(clips, activation_layer="sigmoid", batch_size=192, num_workers=16)["airgun"].values
    s["effnet"] = effnet.predict(clips, activation_layer="sigmoid", batch_size=192, num_workers=16)["airgun"].values

    cand = (s["resnet"] > 0.05) | (s["effnet"] > 0.05)
    s["perch"] = 0.0
    if cand.any():
        emb = np.asarray(perch.embed(clips[cand.values], return_dfs=False, return_preds=False,
                                     num_workers=16, batch_size=64))
        s.loc[cand, "perch"] = shallow.predict_proba(emb)[:, 1]
        del emb                                    # don't keep the embeddings
    print(f"{out.name}: {cand.mean():.0%} of clips went to Perch")
    s.to_csv(out)

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

chunk_AMAR1183.20250514T183024Z.csv: 53% of clips went to Perch


  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/140 [00:00<?, ?it/s]

chunk_AMAR1183.20250516T035024Z.csv: 37% of clips went to Perch


  0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
# Define the root folder with the current files to analyze
root2 = r"/mnt/i/Schalldaten_HABITATWal/2025/STHH1/AMAR/batch5"
all_files = sorted(glob(str(Path(root2) / "*.wav"), recursive=True))

In [ ]:
from pathlib import Path
from opensoundscape.utils import make_clip_df
out_dir = Path("/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/detections_airgun_2025/batch5"); out_dir.mkdir(exist_ok=True)
CHUNK = 100                                         # files per chunk

for c in range(0, len(all_files), CHUNK):
    files = all_files[c:c + CHUNK]
    out = out_dir / f"chunk_{Path(files[0]).stem}.csv"     # named after the first file, stable across reruns
    if out.exists():
        continue                                   # already done
    clips = make_clip_df(files, clip_duration=5)
    if "start_time" in clips.columns:
        clips = clips.set_index(["start_time", "end_time"], append=True)

    s = pd.DataFrame(index=clips.index)
    s["resnet"] = resnet.predict(clips, activation_layer="sigmoid", batch_size=192, num_workers=16)["airgun"].values
    s["effnet"] = effnet.predict(clips, activation_layer="sigmoid", batch_size=192, num_workers=16)["airgun"].values

    cand = (s["resnet"] > 0.05) | (s["effnet"] > 0.05)
    s["perch"] = 0.0
    if cand.any():
        emb = np.asarray(perch.embed(clips[cand.values], return_dfs=False, return_preds=False,
                                     num_workers=16, batch_size=64))
        s.loc[cand, "perch"] = shallow.predict_proba(emb)[:, 1]
        del emb                                    # don't keep the embeddings
    print(f"{out.name}: {cand.mean():.0%} of clips went to Perch")
    s.to_csv(out)

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/273 [00:00<?, ?it/s]

chunk_AMAR1183.20250416T235016Z.csv: 73% of clips went to Perch


  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

... (70 outputs left out)

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/199 [00:00<?, ?it/s]

chunk_AMAR1183.20250513T091023Z.csv: 53% of clips went to Perch
